A digital marketing agency wants to automatically infer the gender of anonymous blog readers/writers to personalize advertising content. Currently, 40-60% of blog visitors provide no demographic data, limiting targeting precision.

Binary text classification: Given a blog post, predict GENDER ∈ {M, F}

Target Audience: Marketing teams (need accurate targeting), data privacy officers (ethical use), content creators (bias awareness)

Data Understanding

In [25]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

In [26]:
#NLP - Libraries
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [27]:
#Sklearn - Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

#Sklearn - Dimensionality Reduction
from sklearn.decomposition import TruncatedSVD

#Sklearn - Models
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

#Sklearn - Evaluation & Utilities
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_validate, GridSearchCV)
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, accuracy_score,
                              f1_score, precision_score, recall_score)
from sklearn.preprocessing import LabelEncoder

#Word2Vec
!pip install gensim
from gensim.models import Word2Vec

In [28]:
#Visualization
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from collections import Counter

print("All libraries imported successfully!")

All libraries imported successfully!


In [29]:

from nltk.corpus import stopwords

#Load Data
df = pd.read_excel("B9AI006 - BLOG GENDER BALANCED.xlsx")
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

#Class Distribution
print(f"\nClass Distribution:\n{df['GENDER'].value_counts()}")
print(f"\nClass Balance Ratio: {df['GENDER'].value_counts(normalize=True).to_dict()}")

#Text Statistics
df['text_length'] = df['BLOG'].astype(str).apply(len)
df['word_count'] = df['BLOG'].astype(str).apply(lambda x: len(x.split()))

print(f"\nText Length Statistics:")
print(df.groupby('GENDER')['text_length'].describe())
print(f"\nWord Count Statistics:")
print(df.groupby('GENDER')['word_count'].describe())

#EDA Visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('PHASE 2: Exploratory Data Analysis — Blog Gender Dataset',
             fontsize=14, fontweight='bold')

# Plot 1: Class Distribution
df['GENDER'].value_counts().plot(kind='bar', ax=axes[0,0], color=['#2196F3','#FF5722'])
axes[0,0].set_title('Class Distribution')
axes[0,0].set_ylabel('Count')
axes[0,0].set_xticklabels(['Female (F)', 'Male (M)'], rotation=0)

# Plot 2: Text Length Distribution by Gender
for gender, color in [('F','#2196F3'), ('M','#FF5722')]:
    subset = df[df['GENDER']==gender]['text_length']
    axes[0,1].hist(subset, bins=50, alpha=0.6, label=gender, color=color)
axes[0,1].set_title('Text Length Distribution by Gender')
axes[0,1].set_xlabel('Character Count')
axes[0,1].legend()
axes[0,1].set_xlim(0, 15000)

# Plot 3: Word Count Distribution (Box Plot)
df.boxplot(column='word_count', by='GENDER', ax=axes[0,2])
axes[0,2].set_title('Word Count by Gender')
axes[0,2].set_ylabel('Word Count')
axes[0,2].set_ylim(0, 3000)
plt.sca(axes[0,2])
plt.xlabel('Gender')

# Plot 4: Top 20 Words - Female
stop_words = set(stopwords.words('english'))
female_text = ' '.join(df[df['GENDER']=='F']['BLOG'].astype(str).tolist()).lower()
female_words = [w for w in female_text.split() if w.isalpha() and w not in stop_words and len(w) > 2]
female_top = Counter(female_words).most_common(20)
axes[1,0].barh([w[0] for w in female_top][::-1], [w[1] for w in female_top][::-1], color='#2196F3')
axes[1,0].set_title('Top 20 Words — Female')

# Plot 5: Top 20 Words — Male
male_text = ' '.join(df[df['GENDER']=='M']['BLOG'].astype(str).tolist()).lower()
male_words = [w for w in male_text.split() if w.isalpha() and w not in stop_words and len(w) > 2]
male_top = Counter(male_words).most_common(20)
axes[1,1].barh([w[0] for w in male_top][::-1], [w[1] for w in male_top][::-1], color='#FF5722')
axes[1,1].set_title('Top 20 Words — Male')

# Plot 6: Average Sentence Length by Gender
df['avg_word_len'] = df['BLOG'].astype(str).apply(
    lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0)
df.boxplot(column='avg_word_len', by='GENDER', ax=axes[1,2])
axes[1,2].set_title('Avg Word Length by Gender')
axes[1,2].set_ylabel('Characters per Word')
plt.sca(axes[1,2])
plt.xlabel('Gender')

plt.tight_layout()
plt.show()
plt.close()


Dataset shape: (2600, 2)
Columns: ['BLOG', 'GENDER']

Data types:
BLOG      object
GENDER    object
dtype: object

Missing values:
BLOG      1
GENDER    0
dtype: int64

Duplicate rows: 11

Class Distribution:
GENDER
F    1300
M    1300
Name: count, dtype: int64

Class Balance Ratio: {'F': 0.5, 'M': 0.5}

Text Length Statistics:
         count         mean          std    min     25%     50%      75%  \
GENDER                                                                     
F       1300.0  2423.047692  4565.105593  155.0  650.75  1141.0  1970.25   
M       1300.0  2570.713846  4836.539175    3.0  637.00  1176.5  2073.25   

            max  
GENDER           
F       32714.0  
M       32714.0  

Word Count Statistics:
         count        mean         std   min    25%    50%    75%     max
GENDER                                                                   
F       1300.0  440.147692  819.041351  26.0  118.0  210.0  353.0  6142.0
M       1300.0  454.780769  853.028348   1.0  

Data Preparation

In [30]:
#Text preprocessing
lemmatizer = WordNetLemmatizer()
stop_words_set = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens
              if w not in stop_words_set and len(w) > 2]
    return ' '.join(tokens)

print("Preprocessing text")
df['clean_text'] = df['BLOG'].apply(preprocess_text)
print(f"Sample cleaned text: {df['clean_text'].iloc[0][:200]}...")

Preprocessing text
Sample cleaned text: beyond getting travel day show today guest post gillian onegiantstepcom sum imperceptible change happens travel start appreciating thing never thought would process maybe even learn new way see world ...


In [31]:
#encode target variable
le = LabelEncoder()
y = le.fit_transform(df['GENDER'])  # F=0, M=1
print(f"\nTarget encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")


Target encoding: {'F': np.int64(0), 'M': np.int64(1)}


In [32]:
#train test split - 80/20
X_text_train, X_text_test, y_train, y_test = train_test_split(
    df['clean_text'], y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train size: {len(X_text_train)}, Test size: {len(X_text_test)}")

Train size: 2080, Test size: 520


In [33]:
#representation 1 - TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),        #Unigrams and Bigrams
    max_features=10000,         #Limit to top 10K features
    sublinear_tf=True,          #Log normalization of TF
    min_df=2,                   #Ignore very rare terms
    max_df=0.95                 #Ignore very common terms
)
X_tfidf_train = tfidf_vectorizer.fit_transform(X_text_train)
X_tfidf_test = tfidf_vectorizer.transform(X_text_test)
print(f"TF-IDF shape: Train={X_tfidf_train.shape}, Test={X_tfidf_test.shape}")

TF-IDF shape: Train=(2080, 10000), Test=(520, 10000)


In [34]:
#representation 2 - TF-IDF + TruncatedSVD
print("\nRepresentation 2: TF-IDF + TruncatedSVD (200 components)")
svd = TruncatedSVD(n_components=200, random_state=42)
X_svd_train = svd.fit_transform(X_tfidf_train)
X_svd_test = svd.transform(X_tfidf_test)
explained_var = svd.explained_variance_ratio_.sum()
print(f"TF-IDF+SVD shape: Train={X_svd_train.shape}, Test={X_svd_test.shape}")
print(f"Explained variance captured: {explained_var:.2%}")


Representation 2: TF-IDF + TruncatedSVD (200 components)
TF-IDF+SVD shape: Train=(2080, 200), Test=(520, 200)
Explained variance captured: 26.25%


In [35]:
#representation 3- Word2Vec
#Tokenize for Word2Vec
train_sentences = [text.split() for text in X_text_train]
test_sentences = [text.split() for text in X_text_test]
all_sentences = [text.split() for text in df['clean_text']]

#Train Word2Vec model on entire corpus
w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=200,            #200-dimensional embeddings
    window=5,                   #Context window of 5 words
    min_count=2,                #Ignore words appearing < 2 times
    workers=4,
    sg=1,                       #Skip-Gram (better for smaller datasets)
    epochs=20
)
print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")

def get_doc_vector(text_tokens, model, vec_size=200):
    vectors = [model.wv[word] for word in text_tokens if word in model.wv]
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vec_size)

X_w2v_train = np.array([get_doc_vector(s, w2v_model) for s in train_sentences])
X_w2v_test = np.array([get_doc_vector(s, w2v_model) for s in test_sentences])
print(f"Word2Vec shape: Train={X_w2v_train.shape}, Test={X_w2v_test.shape}")

Word2Vec vocabulary size: 24129
Word2Vec shape: Train=(2080, 200), Test=(520, 200)


Modelling

In [36]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

all_results = []

def evaluate_model(model, X_train, X_test, y_train, y_test,
                   model_name, repr_name, cv=cv):

    #Cross-validation on training set
    cv_scores = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring=['accuracy', 'f1', 'roc_auc'],
        return_train_score=False
    )

    #Fit on full training set and predict on test set
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    #Get probabilities for AUC (handle models without predict_proba)
    try:
        y_prob = model.predict_proba(X_test)[:, 1]
    except AttributeError:
        try:
            y_prob = model.decision_function(X_test)
        except:
            y_prob = y_pred

    test_auc = roc_auc_score(y_test, y_prob)

    result = {
        'Representation': repr_name,
        'Algorithm': model_name,
        'CV_Accuracy_Mean': cv_scores['test_accuracy'].mean(),
        'CV_Accuracy_Std': cv_scores['test_accuracy'].std(),
        'CV_F1_Mean': cv_scores['test_f1'].mean(),
        'CV_AUC_Mean': cv_scores['test_roc_auc'].mean(),
        'Test_Accuracy': accuracy_score(y_test, y_pred),
        'Test_Precision': precision_score(y_test, y_pred),
        'Test_Recall': recall_score(y_test, y_pred),
        'Test_F1': f1_score(y_test, y_pred),
        'Test_AUC': test_auc,
        'y_pred': y_pred,
        'y_prob': y_prob
    }

    print(f"  {repr_name} + {model_name}: "
          f"CV Acc={result['CV_Accuracy_Mean']:.3f}±{result['CV_Accuracy_Std']:.3f}, "
          f"Test Acc={result['Test_Accuracy']:.3f}, "
          f"Test AUC={result['Test_AUC']:.3f}")

    return result

In [37]:
#Navie Bayes Algorithm
#NB + TF-IDF
nb_tfidf = MultinomialNB(alpha=0.5)
r = evaluate_model(nb_tfidf, X_tfidf_train, X_tfidf_test, y_train, y_test,
                   'Naive Bayes', 'TF-IDF')
all_results.append(r)

#NB + TF-IDF+SVD
nb_svd = GaussianNB()
r = evaluate_model(nb_svd, X_svd_train, X_svd_test, y_train, y_test,
                   'Naive Bayes', 'TF-IDF+SVD')
all_results.append(r)

#NB + Word2Vec
nb_w2v = GaussianNB()
r = evaluate_model(nb_w2v, X_w2v_train, X_w2v_test, y_train, y_test,
                   'Naive Bayes', 'Word2Vec')
all_results.append(r)

  TF-IDF + Naive Bayes: CV Acc=0.705±0.025, Test Acc=0.698, Test AUC=0.781
  TF-IDF+SVD + Naive Bayes: CV Acc=0.648±0.041, Test Acc=0.587, Test AUC=0.742
  Word2Vec + Naive Bayes: CV Acc=0.650±0.031, Test Acc=0.700, Test AUC=0.765


In [38]:
#SVM
#SVM + TF-IDF (LinearSVC wrapped for probability estimates)
svm_tfidf = CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=10000), cv=5)
r = evaluate_model(svm_tfidf, X_tfidf_train, X_tfidf_test, y_train, y_test,
                   'SVM', 'TF-IDF')
all_results.append(r)

#SVM + TF-IDF+SVD (RBF kernel for dense low-dimensional data)
svm_svd = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
r = evaluate_model(svm_svd, X_svd_train, X_svd_test, y_train, y_test,
                   'SVM', 'TF-IDF+SVD')
all_results.append(r)

#SVM + Word2Vec
svm_w2v = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
r = evaluate_model(svm_w2v, X_w2v_train, X_w2v_test, y_train, y_test,
                   'SVM', 'Word2Vec')
all_results.append(r)

  TF-IDF + SVM: CV Acc=0.713±0.028, Test Acc=0.681, Test AUC=0.762
  TF-IDF+SVD + SVM: CV Acc=0.704±0.023, Test Acc=0.700, Test AUC=0.775
  Word2Vec + SVM: CV Acc=0.696±0.031, Test Acc=0.708, Test AUC=0.781


In [39]:
#Random Forest
#RF + TF-IDF
rf_tfidf = RandomForestClassifier(n_estimators=200, max_depth=20,
                                   min_samples_split=5, random_state=42, n_jobs=-1)
r = evaluate_model(rf_tfidf, X_tfidf_train.toarray(), X_tfidf_test.toarray(),
                   y_train, y_test, 'Random Forest', 'TF-IDF')
all_results.append(r)

#RF + TF-IDF+SVD
rf_svd = RandomForestClassifier(n_estimators=200, max_depth=20,
                                 min_samples_split=5, random_state=42, n_jobs=-1)
r = evaluate_model(rf_svd, X_svd_train, X_svd_test, y_train, y_test,
                   'Random Forest', 'TF-IDF+SVD')
all_results.append(r)

#RF + Word2Vec
rf_w2v = RandomForestClassifier(n_estimators=200, max_depth=20,
                                 min_samples_split=5, random_state=42, n_jobs=-1)
r = evaluate_model(rf_w2v, X_w2v_train, X_w2v_test, y_train, y_test,
                   'Random Forest', 'Word2Vec')
all_results.append(r)

  TF-IDF + Random Forest: CV Acc=0.689±0.032, Test Acc=0.665, Test AUC=0.752
  TF-IDF+SVD + Random Forest: CV Acc=0.689±0.020, Test Acc=0.677, Test AUC=0.753
  Word2Vec + Random Forest: CV Acc=0.689±0.025, Test Acc=0.692, Test AUC=0.776


Evaluation

In [40]:
#Master results table
results_df = pd.DataFrame(all_results)
display_cols = ['Representation', 'Algorithm', 'CV_Accuracy_Mean', 'CV_Accuracy_Std',
                'CV_F1_Mean', 'CV_AUC_Mean', 'Test_Accuracy', 'Test_F1', 'Test_AUC']
results_table = results_df[display_cols].copy()
results_table.columns = ['Representation', 'Algorithm', 'CV Acc (Mean)', 'CV Acc (SD)',
                          'CV F1', 'CV AUC', 'Test Acc', 'Test F1', 'Test AUC']

print("\nMASTER RESULTS TABLE:")
print("="*100)
print(results_table.to_string(index=False, float_format='%.3f'))
print("="*100)

#Save results to CSV
results_table.to_csv('task2_results.csv', index=False)
print("\n Results saved to 'task2_results.csv'")


MASTER RESULTS TABLE:
Representation     Algorithm  CV Acc (Mean)  CV Acc (SD)  CV F1  CV AUC  Test Acc  Test F1  Test AUC
        TF-IDF   Naive Bayes          0.705        0.025  0.680   0.776     0.698    0.688     0.781
    TF-IDF+SVD   Naive Bayes          0.648        0.041  0.655   0.705     0.587    0.703     0.742
      Word2Vec   Naive Bayes          0.650        0.031  0.613   0.719     0.700    0.685     0.765
        TF-IDF           SVM          0.713        0.028  0.714   0.780     0.681    0.687     0.762
    TF-IDF+SVD           SVM          0.704        0.023  0.708   0.783     0.700    0.734     0.775
      Word2Vec           SVM          0.696        0.031  0.689   0.764     0.708    0.709     0.781
        TF-IDF Random Forest          0.689        0.032  0.673   0.767     0.665    0.663     0.752
    TF-IDF+SVD Random Forest          0.689        0.020  0.680   0.754     0.677    0.711     0.753
      Word2Vec Random Forest          0.689        0.025  0.683   0.

In [41]:
#AI Studio benchmark comparison
print("\n AI STUDIO BENCHMARK COMPARISON:")
print("-"*60)
benchmarks = {'NB': {'Accuracy': 0.81, 'AUC': 0.92},
              'GLM': {'Accuracy': 0.64, 'AUC': 0.71},
              'DL': {'Accuracy': 0.75, 'AUC': 0.83}}

best_result = results_df.loc[results_df['Test_AUC'].idxmax()]
print(f"Best Model: {best_result['Representation']} + {best_result['Algorithm']}")
print(f"  Test Accuracy: {best_result['Test_Accuracy']:.3f} (Benchmark NB: 0.810)")
print(f"  Test AUC:      {best_result['Test_AUC']:.3f} (Benchmark NB: 0.920)")

if best_result['Test_Accuracy'] >= 0.81:
    print("  EXCEEDS AI Studio NB benchmark accuracy!")
else:
    print(f"  Below NB benchmark by {0.81 - best_result['Test_Accuracy']:.3f}")



 AI STUDIO BENCHMARK COMPARISON:
------------------------------------------------------------
Best Model: TF-IDF + Naive Bayes
  Test Accuracy: 0.698 (Benchmark NB: 0.810)
  Test AUC:      0.781 (Benchmark NB: 0.920)
  Below NB benchmark by 0.112


AI Used for plots visualization

In [42]:
#Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('PHASE 5: Model Evaluation Results', fontsize=14, fontweight='bold')

# Plot 1: Accuracy Comparison Bar Chart
plot_data = results_df[['Representation', 'Algorithm', 'Test_Accuracy']].copy()
plot_data['Label'] = plot_data['Representation'] + '\n+ ' + plot_data['Algorithm']
colors = ['#2196F3' if 'TF-IDF+SVD' in r else '#FF5722' if 'Word2Vec' in r else '#4CAF50'
          for r in plot_data['Representation']]
bars = axes[0,0].bar(range(len(plot_data)), plot_data['Test_Accuracy'], color=colors)
axes[0,0].axhline(y=0.81, color='red', linestyle='--', linewidth=1.5, label='AI Studio NB (0.81)')
axes[0,0].axhline(y=0.75, color='orange', linestyle='--', linewidth=1, label='AI Studio DL (0.75)')
axes[0,0].set_xticks(range(len(plot_data)))
axes[0,0].set_xticklabels(plot_data['Label'], rotation=45, ha='right', fontsize=7)
axes[0,0].set_ylabel('Test Accuracy')
axes[0,0].set_title('Test Accuracy — All Models vs AI Studio Benchmark')
axes[0,0].legend(fontsize=8)
axes[0,0].set_ylim(0.4, 1.0)

# Plot 2: AUC Comparison
bars2 = axes[0,1].bar(range(len(plot_data)), results_df['Test_AUC'], color=colors)
axes[0,1].axhline(y=0.92, color='red', linestyle='--', linewidth=1.5, label='AI Studio NB (0.92)')
axes[0,1].axhline(y=0.83, color='orange', linestyle='--', linewidth=1, label='AI Studio DL (0.83)')
axes[0,1].set_xticks(range(len(plot_data)))
axes[0,1].set_xticklabels(plot_data['Label'], rotation=45, ha='right', fontsize=7)
axes[0,1].set_ylabel('Test AUC')
axes[0,1].set_title('Test AUC — All Models vs AI Studio Benchmark')
axes[0,1].legend(fontsize=8)
axes[0,1].set_ylim(0.4, 1.0)

# Plot 3: ROC Curves (top 3 models by AUC)
top3 = results_df.nlargest(3, 'Test_AUC')
colors_roc = ['#2196F3', '#FF5722', '#4CAF50']
for i, (_, row) in enumerate(top3.iterrows()):
    fpr, tpr, _ = roc_curve(y_test, row['y_prob'])
    axes[1,0].plot(fpr, tpr, color=colors_roc[i], linewidth=2,
                   label=f"{row['Representation']}+{row['Algorithm']} (AUC={row['Test_AUC']:.3f})")
axes[1,0].plot([0,1], [0,1], 'k--', linewidth=0.8)
axes[1,0].set_xlabel('False Positive Rate')
axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].set_title('ROC Curves — Top 3 Models')
axes[1,0].legend(fontsize=8)

# Plot 4: Confusion Matrix of Best Model
best_idx = results_df['Test_AUC'].idxmax()
cm = confusion_matrix(y_test, results_df.loc[best_idx, 'y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1,1],
            xticklabels=['Female', 'Male'], yticklabels=['Female', 'Male'])
axes[1,1].set_xlabel('Predicted')
axes[1,1].set_ylabel('Actual')
axes[1,1].set_title(f"Confusion Matrix — Best Model\n"
                     f"({results_df.loc[best_idx, 'Representation']} + "
                     f"{results_df.loc[best_idx, 'Algorithm']})")

plt.tight_layout()
plt.savefig('evaluation.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n Evaluation visualizations saved to 'evaluation.png'")


 Evaluation visualizations saved to 'evaluation.png'


In [43]:
#Classification report + feature importance
#Classification Report for Best Model
best = results_df.loc[results_df['Test_AUC'].idxmax()]
print(f"\n CLASSIFICATION REPORT — Best Model ({best['Representation']} + {best['Algorithm']}):")
print(classification_report(y_test, best['y_pred'], target_names=['Female', 'Male']))

#Feature Importance (from TF-IDF + Random Forest)
print("\n TOP 20 MOST PREDICTIVE FEATURES (TF-IDF + Random Forest):")
feature_names = tfidf_vectorizer.get_feature_names_out()
rf_fitted = rf_tfidf  # Already fitted
importances = rf_fitted.feature_importances_
top_indices = np.argsort(importances)[-20:][::-1]
print("-"*40)
for i, idx in enumerate(top_indices, 1):
    print(f"  {i:2d}. {feature_names[idx]:20s} — importance: {importances[idx]:.4f}")

#Feature importance plot
plt.figure(figsize=(10, 6))
top_features = [(feature_names[i], importances[i]) for i in top_indices]
plt.barh([f[0] for f in top_features][::-1], [f[1] for f in top_features][::-1],
         color='#2196F3')
plt.xlabel('Feature Importance')
plt.title('Top 20 Most Predictive Features (TF-IDF + Random Forest)')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.close()
print(" Feature importance plot saved to 'feature_importance.png'")


 CLASSIFICATION REPORT — Best Model (TF-IDF + Naive Bayes):
              precision    recall  f1-score   support

      Female       0.69      0.73      0.71       260
        Male       0.71      0.67      0.69       260

    accuracy                           0.70       520
   macro avg       0.70      0.70      0.70       520
weighted avg       0.70      0.70      0.70       520


 TOP 20 MOST PREDICTIVE FEATURES (TF-IDF + Random Forest):
----------------------------------------
   1. love                 — importance: 0.0106
   2. mom                  — importance: 0.0064
   3. husband              — importance: 0.0060
   4. day                  — importance: 0.0059
   5. really               — importance: 0.0058
   6. food                 — importance: 0.0054
   7. little               — importance: 0.0050
   8. john                 — importance: 0.0049
   9. kid                  — importance: 0.0047
  10. make                 — importance: 0.0042
  11. game                 — im

Deployment

In [44]:
# Save model example
import joblib
best_model_name = f"{best['Representation']}_{best['Algorithm']}"
# Save the best pipeline components
# joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')
# joblib.dump(best_model, 'best_classifier.pkl')
print(f"Best model ({best_model_name}) ready for deployment.")

Best model (TF-IDF_Naive Bayes) ready for deployment.


In [45]:
print(f"\nTotal experiments run: {len(all_results)}")
print(f"Representations used: TF-IDF, TF-IDF+SVD, Word2Vec")
print(f"Algorithms used: Naive Bayes, SVM, Random Forest")
print(f"Best model: {best['Representation']} + {best['Algorithm']}")
print(f"   Test Accuracy: {best['Test_Accuracy']:.3f}")
print(f"   Test F1-Score: {best['Test_F1']:.3f}")
print(f"   Test AUC:      {best['Test_AUC']:.3f}")


Total experiments run: 9
Representations used: TF-IDF, TF-IDF+SVD, Word2Vec
Algorithms used: Naive Bayes, SVM, Random Forest
Best model: TF-IDF + Naive Bayes
   Test Accuracy: 0.698
   Test F1-Score: 0.688
   Test AUC:      0.781
